In [23]:
import io
import zipfile
import requests
import pandas as pd
from urllib.parse import unquote

# ==================================================
# 1. 인증키와 조회 기간
# ==================================================

# 서비스명:
# 행정안전부_행정동별(통반단위) 성/연령별 주민등록 인구수
# 앞서 정상 조회에 성공했던 인증키를 넣으세요.
population_serviceKey = unquote(
    '6ZlAcr4RuErZhSgB4zE9YaHbbfiQes2YrtLoy3YvjYZ6D8tTehdXARkpHVMZAJRDumjiZ5WvLsZ%2F72xylOI4Zg%3D%3D'.strip()
)

# 법정동 코드 API 인증키는 사용하지 않습니다.

start_month = '2022-10'
end_month = '2022-10'

month_list = pd.period_range(
    start=start_month,
    end=end_month,
    freq='M'
).strftime('%Y%m')

population_url = (
    'https://apis.data.go.kr/1741000/admmSexdAgePpltn'
    '/selectAdmmSexdAgePpltn'
)

# 2026-09-30 기준 공식 행정동 코드 목록
# 과거 지역 조회를 위해 말소코드 포함 파일 사용
# 인증키가 필요하지 않습니다.
code_url = (
    'https://www.mois.go.kr/cmm/fms/FileDown.do'
    '?atchFileId=FILE_00149355_BR8t5D&fileSn=2'
)

df_list = []
query_log = []

with requests.Session() as session:

    # ==================================================
    # 2. 공식 행정동 코드 목록 읽기
    # ==================================================

    code_response = session.get(code_url, timeout=60)
    code_response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(code_response.content)) as archive:
        excel_name = next(
            name for name in archive.namelist()
            if 'KIKCD_H' in name.upper()
            and name.lower().endswith('.xlsx')
        )

        df_admm = pd.read_excel(
            io.BytesIO(archive.read(excel_name)),
            dtype=str
        )

    df_admm.columns = df_admm.columns.str.strip()

    df_admm['행정동코드'] = (
        df_admm['행정동코드']
        .str.strip()
        .str.zfill(10)
    )

    df_admm['_생성일'] = pd.to_datetime(
        df_admm['생성일자'].str.strip(),
        format='%Y%m%d',
        errors='coerce'
    )

    df_admm['_말소일'] = pd.to_datetime(
        df_admm['말소일자'].str.strip(),
        format='%Y%m%d',
        errors='coerce'
    )

    # 읍면동만 선택: 시도·시군구 집계 코드 제외
    mask = (
        df_admm['읍면동명'].fillna('').str.strip().ne('')
        & df_admm['행정동코드'].str.fullmatch(r'\d{10}', na=False)
    )

    df_admm = df_admm.loc[mask].copy()

    df_admm['지역주소명'] = (
        df_admm[['시도명', '시군구명', '읍면동명']]
        .fillna('')
        .agg(' '.join, axis=1)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    print(f'행정동 코드 목록(말소 포함): {len(df_admm):,}건')

    # ==================================================
    # 3. 인구 API 요청 조건
    # ==================================================

    params = {
        'serviceKey': population_serviceKey,
        'admmCd': '',
        'srchFrYm': '',
        'srchToYm': '',
        'lv': '4',
        'regSeCd': '1',
        'type': 'JSON',
        'numOfRows': '100',
        'pageNo': '1'
    }

    # ==================================================
    # 4. 월 → 지역 → 페이지 반복
    # ==================================================

    for month in month_list:
        month_start = pd.to_datetime(month, format='%Y%m')
        month_end = month_start + pd.offsets.MonthEnd(0)

        # 해당 월과 존속 기간이 겹치는 행정동 선택
        mask = (
            (
                df_admm['_생성일'].isna()
                | df_admm['_생성일'].le(month_end)
            )
            & (
                df_admm['_말소일'].isna()
                | df_admm['_말소일'].ge(month_start)
            )
        )

        df_region = (
            df_admm.loc[mask, ['행정동코드', '지역주소명']]
            .rename(columns={'행정동코드': '지역코드'})
            .drop_duplicates(subset='지역코드')
            .reset_index(drop=True)
        )

        params['srchFrYm'] = month
        params['srchToYm'] = month

        print(f'\n{month} | 조회 대상 행정동: {len(df_region):,}개')

        for region_no, (region_cd, region_name) in enumerate(
            df_region.itertuples(index=False, name=None),
            start=1
        ):
            params['admmCd'] = region_cd
            page_no = 1
            region_count = 0

            while True:
                params['pageNo'] = str(page_no)

                response = session.get(
                    population_url,
                    params=params,
                    timeout=60
                )

                if response.status_code != 200:
                    raise RuntimeError(
                        f'{month} | {region_cd} | {region_name}\n'
                        f'HTTP {response.status_code}\n'
                        f'{response.text[:1000]}'
                    )

                data = response.json()
                result = data.get('Response')

                if not isinstance(result, dict):
                    raise RuntimeError(
                        f'{month} | {region_cd} | 응답 구조 오류\n'
                        f'{response.text[:1000]}'
                    )

                head = result['head']
                result_code = str(head['resultCode'])

                # 데이터 없는 지역은 건너뛰기
                if (
                    result_code == '3'
                    and head.get('resultMsg') == 'NODATA_ERROR'
                ):
                    if region_count > 0:
                        raise RuntimeError(
                            f'{month} | {region_cd}: '
                            '일부 페이지를 읽은 뒤 데이터 없음이 반환됐습니다.'
                        )
                    break

                # 인증·호출 한도 등 다른 오류는 중단
                if result_code != '0':
                    raise RuntimeError(
                        f'{month} | {region_cd} | API 오류: {head}'
                    )

                total_count = int(head['totalCount'])

                if total_count == 0:
                    break

                items = (result.get('items') or {}).get('item') or []

                if isinstance(items, dict):
                    items = [items]

                if not items:
                    raise RuntimeError(
                        f'{month} | {region_cd} | 페이지 {page_no}: '
                        '전체 건수는 있으나 항목이 없습니다.'
                    )

                df_list.append(pd.DataFrame(items))
                region_count += len(items)

                print(
                    f'{month} | 지역 {region_no}/{len(df_region)} '
                    f'| {region_name} | 페이지 {page_no} '
                    f'| {region_count:,}/{total_count:,}건'
                )

                if region_count >= total_count:
                    break

                page_no += 1

            query_log.append({
                '기준연월': month,
                '요청지역코드': region_cd,
                '요청지역명': region_name,
                '조회건수': region_count,
                '조회상태': '성공' if region_count else '데이터 없음'
            })

            if region_count == 0:
                print(
                    f'{month} | 지역 {region_no}/{len(df_region)} '
                    f'| {region_name} | 데이터 없음 → 건너뛰기'
                )

# ==================================================
# 5. 전체 데이터 합치기
# ==================================================

df = (
    pd.concat(df_list, ignore_index=True)
    if df_list
    else pd.DataFrame()
)

df_query_log = pd.DataFrame(
    query_log,
    columns=[
        '기준연월', '요청지역코드', '요청지역명',
        '조회건수', '조회상태'
    ]
)

# ==================================================
# 6. 숫자 변환과 한글 컬럼명
# ==================================================

if not df.empty:
    quantity_columns = [
        col for col in df.columns
        if col.endswith('NmprCnt')
    ]

    df[quantity_columns] = df[quantity_columns].apply(
        pd.to_numeric,
        errors='raise'
    )

    rename_columns = {
        'statsYm': '기준연월',
        'admmCd': '행정기관코드',
        'ctpvNm': '시도',
        'sggNm': '시군구',
        'dongNm': '읍면동',
        'tong': '통',
        'ban': '반',
        'totNmprCnt': '총인구수',
        'maleNmprCnt': '남자인구수',
        'femlNmprCnt': '여자인구수'
    }

    for age in range(0, 101, 10):
        age_label = (
            f'{age}~{age + 9}세'
            if age < 100
            else '100세이상'
        )

        rename_columns[f'male{age}AgeNmprCnt'] = f'남자_{age_label}'
        rename_columns[f'feml{age}AgeNmprCnt'] = f'여자_{age_label}'

    df = df.rename(columns=rename_columns)

    ordered_columns = [
        col for col in rename_columns.values()
        if col in df.columns
    ]

    df = df[
        ordered_columns
        + [col for col in df.columns if col not in ordered_columns]
    ]

    # 중복은 자동 삭제하지 않고 확인
    key_columns = ['기준연월', '행정기관코드', '통', '반']

    duplicate_count = df.duplicated(
        subset=key_columns,
        keep=False
    ).sum()

    print(f'\n중복 키에 해당하는 행: {duplicate_count:,}건')

    print('\n조회된 시도:')
    print(df['시도'].unique())

    print('\n월별 데이터 건수:')
    print(df.groupby('기준연월').size())

print(f'\n조회 기간: {start_month} ~ {end_month}')
print(f'최종 데이터 크기: {df.shape}')

print('\n조회 상태별 지역·월 수:')
print(df_query_log['조회상태'].value_counts())

df.head()

행정동 코드 목록(말소 포함): 9,034건

202210 | 조회 대상 행정동: 3,582개
202210 | 지역 1/3582 | 서울특별시 종로구 청운효자동 | 페이지 1 | 97/97건
202210 | 지역 2/3582 | 서울특별시 종로구 사직동 | 페이지 1 | 68/68건
202210 | 지역 3/3582 | 서울특별시 종로구 삼청동 | 페이지 1 | 42/42건
202210 | 지역 4/3582 | 서울특별시 종로구 부암동 | 페이지 1 | 99/99건
202210 | 지역 5/3582 | 서울특별시 종로구 평창동 | 페이지 1 | 100/148건
202210 | 지역 5/3582 | 서울특별시 종로구 평창동 | 페이지 2 | 148/148건
202210 | 지역 6/3582 | 서울특별시 종로구 무악동 | 페이지 1 | 71/71건
202210 | 지역 7/3582 | 서울특별시 종로구 교남동 | 페이지 1 | 100/103건
202210 | 지역 7/3582 | 서울특별시 종로구 교남동 | 페이지 2 | 103/103건
202210 | 지역 8/3582 | 서울특별시 종로구 가회동 | 페이지 1 | 58/58건
202210 | 지역 9/3582 | 서울특별시 종로구 종로1.2.3.4가동 | 페이지 1 | 67/67건
202210 | 지역 10/3582 | 서울특별시 종로구 종로5.6가동 | 페이지 1 | 65/65건
202210 | 지역 11/3582 | 서울특별시 종로구 이화동 | 페이지 1 | 93/93건
202210 | 지역 12/3582 | 서울특별시 종로구 혜화동 | 페이지 1 | 100/188건
202210 | 지역 12/3582 | 서울특별시 종로구 혜화동 | 페이지 2 | 188/188건
202210 | 지역 13/3582 | 서울특별시 종로구 창신제1동 | 페이지 1 | 67/67건
202210 | 지역 14/3582 | 서울특별시 종로구 창신제2동 | 페이지 1 | 100/105건
202210 | 지역 14/3582 | 서울특

KeyboardInterrupt: 

In [1]:
import requests

# API 명세서에 나온 최종 요청 주소
url = 'https://apis.data.go.kr/1741000/admmSexdAgePpltn'

# 실제 항목 이름은 API 명세서에 맞춰 작성
params = {
    'serviceKey': '6ZlAcr4RuErZhSgB4zE9YaHbbfiQes2YrtLoy3YvjYZ6D8tTehdXARkpHVMZAJRDumjiZ5WvLsZ%2F72xylOI4Zg%3D%3D'
}

response = requests.get(url, params=params, timeout=30)

print(response.status_code)
print(response.text[:1000])

400
<?xml version="1.0" encoding="UTF-8"?>
<OpenAPI_ServiceResponse>
<cmmMsgHeader>
  <errMsg>NO_OPENAPI_SERVICE_ERROR</errMsg>
  <returnAuthMsg>해당 오픈API 서비스가 없거나 폐기됨</returnAuthMsg>
  <returnReasonCode>12</returnReasonCode>
</cmmMsgHeader>
</OpenAPI_ServiceResponse>



In [2]:
import requests

# 상세기능에 표시된 최종 요청주소를 입력
url = 'https://apis.data.go.kr/1741000/admmSexdAgePpltn'

params = {
    'serviceKey': '6ZlAcr4RuErZhSgB4zE9YaHbbfiQes2YrtLoy3YvjYZ6D8tTehdXARkpHVMZAJRDumjiZ5WvLsZ%2F72xylOI4Zg%3D%3D',
    'admmCd': '1111054000',
    'srchFrYm': '202210',
    'srchToYm': '202210',
    'lv': '4',
    'regSeCd': '1',
    'type': 'JSON',
    'numOfRows': '100',
    'pageNo': '1'
}

response = requests.get(url, params=params, timeout=30)

print(f'상태 코드: {response.status_code}')
print(response.text[:2000])

상태 코드: 400
{
  "OpenAPI_ServiceResponse": {
    "cmmMsgHeader": {
      "errMsg": "NO_OPENAPI_SERVICE_ERROR",
      "returnAuthMsg": "해당 오픈API 서비스가 없거나 폐기됨",
      "returnReasonCode": "12"
    }
  }
}



In [4]:
import requests

url = (
    'https://apis.data.go.kr/1741000/admmSexdAgePpltn'
    '/selectAdmmSexdAgePpltn'
)

params = {
    'serviceKey': '6ZlAcr4RuErZhSgB4zE9YaHbbfiQes2YrtLoy3YvjYZ6D8tTehdXARkpHVMZAJRDumjiZ5WvLsZ%2F72xylOI4Zg%3D%3D',
    'admmCd': '1111054000',
    'srchFrYm': '202210',
    'srchToYm': '202210',
    'lv': '4',
    'regSeCd': '1',
    'type': 'JSON',
    'numOfRows': '100',
    'pageNo': '1'
}

response = requests.get(url, params=params, timeout=30)

print(f'상태 코드: {response.status_code}')
print(response.text[:2000])

상태 코드: 403
{
  "OpenAPI_ServiceResponse": {
    "cmmMsgHeader": {
      "errMsg": "SERVICE_KEY_IS_NOT_REGISTERED_ERROR",
      "returnAuthMsg": "등록되지 않은 서비스키",
      "returnReasonCode": "30"
    }
  }
}



In [7]:
import requests
from urllib.parse import unquote

url = (
    'https://apis.data.go.kr/1741000/admmSexdAgePpltn'
    '/selectAdmmSexdAgePpltn'
)

serviceKey = '6ZlAcr4RuErZhSgB4zE9YaHbbfiQes2YrtLoy3YvjYZ6D8tTehdXARkpHVMZAJRDumjiZ5WvLsZ%2F72xylOI4Zg%3D%3D'

# 앞뒤 공백 제거 + Encoding 키라면 디코딩
serviceKey = unquote(serviceKey.strip())

params = {
    'serviceKey': serviceKey,
    'admmCd': '1111054000',
    'srchFrYm': '202210',
    'srchToYm': '202210',
    'lv': '4',
    'regSeCd': '1',
    'type': 'JSON',
    'numOfRows': '100',
    'pageNo': '1'
}

response = requests.get(url, params=params, timeout=30)

print(f'상태 코드: {response.status_code}')
print(response.text[:2000])

상태 코드: 200
{"Response": {
    "head": {
        "pageNo": "1",
        "resultCode": "0",
        "totalCount": "42",
        "numOfRows": "100",
        "resultMsg": "NORMAL_SERVICE"
    },
    "items": {"item": [
        {
            "tong": "1",
            "male50AgeNmprCnt": "3",
            "feml40AgeNmprCnt": "7",
            "male80AgeNmprCnt": "3",
            "male20AgeNmprCnt": "2",
            "ban": "1",
            "totNmprCnt": "60",
            "feml0AgeNmprCnt": "1",
            "male40AgeNmprCnt": "4",
            "ctpvNm": "서울특별시",
            "male90AgeNmprCnt": "0",
            "dongNm": "삼청동",
            "feml10AgeNmprCnt": "1",
            "feml50AgeNmprCnt": "2",
            "male10AgeNmprCnt": "4",
            "feml100AgeNmprCnt": "0",
            "feml90AgeNmprCnt": "0",
            "male30AgeNmprCnt": "2",
            "feml20AgeNmprCnt": "2",
            "feml60AgeNmprCnt": "5",
            "femlNmprCnt": "29",
            "feml30AgeNmprCnt": "3",
         

In [16]:
import requests
import pandas as pd
from urllib.parse import unquote

url = (
    'https://apis.data.go.kr/1741000/admmSexdAgePpltn'
    '/selectAdmmSexdAgePpltn'
)

# 정상 조회에 성공했던 인증키를 넣으세요.
serviceKey = unquote('6ZlAcr4RuErZhSgB4zE9YaHbbfiQes2YrtLoy3YvjYZ6D8tTehdXARkpHVMZAJRDumjiZ5WvLsZ%2F72xylOI4Zg%3D%3D'.strip())

start_month = '2022-10'

# 아직 집계되지 않았을 수 있는 이번 달은 제외
# end_month = (
#     pd.Timestamp.now(tz='Asia/Seoul').to_period('M') - 1
# ).strftime('%Y-%m')

end_month = '2022-10'

month_list = pd.period_range(
    start=start_month,
    end=end_month,
    freq='M'
).strftime('%Y%m')

params = {
    'serviceKey': serviceKey,
    'admmCd': '0000000000',
    'srchFrYm': '',
    'srchToYm': '',
    'lv': '4',
    'regSeCd': '1',
    'type': 'JSON',
    'numOfRows': '100',
    'pageNo': '1'
}

df_list = []

with requests.Session() as session:
    for month in month_list:
        params['srchFrYm'] = month
        params['srchToYm'] = month

        page_no = 1
        month_count = 0

        while True:
            params['pageNo'] = str(page_no)

            response = session.get(
                url,
                params=params,
                timeout=30
            )

            # 오류가 발생하면 해당 월과 페이지를 확인
            if response.status_code != 200:
                raise RuntimeError(
                    f'{month}, {page_no}페이지 조회 실패\n'
                    f'HTTP {response.status_code}\n'
                    f'{response.text[:1000]}'
                )

            data = response.json()
            result = data.get('Response')

            if not isinstance(result, dict):
                raise RuntimeError(
                    f'{month}, {page_no}페이지 응답 구조 오류\n'
                    f'{response.text[:1000]}'
                )

            head = result['head']

            if str(head['resultCode']) != '0':
                raise RuntimeError(
                    f'{month}, {page_no}페이지 API 오류: {head}'
                )

            total_count = int(head['totalCount'])

            if total_count == 0:
                print(f'{month}: 데이터 없음')
                break

            items = (result.get('items') or {}).get('item') or []

            # 결과가 한 건이면 딕셔너리로 반환되는 경우 처리
            if isinstance(items, dict):
                items = [items]

            if not items:
                raise RuntimeError(
                    f'{month}, {page_no}페이지: '
                    f'전체 건수는 {total_count}인데 데이터가 없습니다.'
                )

            df_list.append(pd.DataFrame(items))
            month_count += len(items)

            print(
                f'{month} | 페이지 {page_no} | '
                f'읽은 건수 {month_count:,}/{total_count:,}'
            )

            if month_count >= total_count:
                break

            page_no += 1

# 모든 월과 페이지를 하나의 데이터프레임으로 합치기
df = (
    pd.concat(df_list, ignore_index=True)
    if df_list
    else pd.DataFrame()
)

# 인구수 컬럼을 숫자로 변환
quantity_columns = [
    col for col in df.columns
    if col.endswith('NmprCnt')
]

df[quantity_columns] = df[quantity_columns].apply(
    pd.to_numeric,
    errors='raise'
)

print(f'\n조회 기간: {start_month} ~ {end_month}')
print(f'최종 데이터 크기: {df.shape}')

if not df.empty:
    print('\n월별 데이터 건수:')
    print(df.groupby('statsYm').size())

df.head().to_clipboard(index=False)

RuntimeError: 202210, 1페이지 API 오류: {'pageNo': '1', 'resultCode': '3', 'totalCount': '0', 'numOfRows': '100', 'resultMsg': 'NODATA_ERROR'}

In [13]:
df.head().to_clipboard(index=False)

In [14]:
# 기본 컬럼명
rename_columns = {
    'statsYm': '기준연월',
    'admmCd': '행정기관코드',
    'ctpvNm': '시도',
    'sggNm': '시군구',
    'dongNm': '읍면동',
    'tong': '통',
    'ban': '반',
    'totNmprCnt': '총인구수',
    'maleNmprCnt': '남자인구수',
    'femlNmprCnt': '여자인구수'
}

# 연령대 컬럼명 자동 생성
for age in range(0, 101, 10):
    age_label = (
        f'{age}~{age + 9}세'
        if age < 100
        else '100세이상'
    )

    rename_columns[f'male{age}AgeNmprCnt'] = f'남자_{age_label}'
    rename_columns[f'feml{age}AgeNmprCnt'] = f'여자_{age_label}'

# 컬럼명 변경
df = df.rename(columns=rename_columns)

# 위 딕셔너리에 정의한 순서로 컬럼 정렬
df = df[
    [col for col in rename_columns.values() if col in df.columns]
    + [col for col in df.columns if col not in rename_columns.values()]
]

df.head()

,기준연월,행정기관코드,시도,시군구,읍면동,통,반,총인구수,남자인구수,여자인구수,...,남자_60~69세,여자_60~69세,남자_70~79세,여자_70~79세,남자_80~89세,여자_80~89세,남자_90~99세,여자_90~99세,남자_100세이상,여자_100세이상
0,202210,1111054000,서울특별시,종로구,삼청동,1,1,60,31,29,...,5,5,4,5,3,3,0,0,0,0
1,202210,1111054000,서울특별시,종로구,삼청동,1,2,27,9,18,...,1,1,1,4,0,2,0,1,0,0
2,202210,1111054000,서울특별시,종로구,삼청동,1,3,50,21,29,...,4,7,3,1,0,4,1,0,0,0
3,202210,1111054000,서울특별시,종로구,삼청동,1,4,41,19,22,...,2,2,2,2,2,0,1,1,0,0
4,202210,1111054000,서울특별시,종로구,삼청동,2,1,53,37,16,...,2,4,1,1,0,2,0,0,0,0


In [15]:
df['시군구'].unique()

array(['종로구'], dtype=object)

In [ ]:
import requests
from urllib.parse import unquote

url = (
    'https://apis.data.go.kr/1741000/admmSexdAgePpltn'
    '/selectAdmmSexdAgePpltn'
)

serviceKey = unquote('여기에 실제 인증키 붙여넣기'.strip())

params = {
    'serviceKey': serviceKey,
    'admmCd': '0000000000',  # 전국 코드로 지원되는지 확인 필요
    'srchFrYm': '202210',
    'srchToYm': '202210',
    'lv': '4',
    'regSeCd': '1',
    'type': 'JSON',
    'numOfRows': '100',
    'pageNo': '1'
}

response = requests.get(url, params=params, timeout=30)

print(f'상태 코드: {response.status_code}')
print(response.text[:2000])

In [17]:
import requests
import pandas as pd
from urllib.parse import unquote

url = (
    'https://apis.data.go.kr/1741000/admmSexdAgePpltn'
    '/selectAdmmSexdAgePpltn'
)

serviceKey = unquote('6ZlAcr4RuErZhSgB4zE9YaHbbfiQes2YrtLoy3YvjYZ6D8tTehdXARkpHVMZAJRDumjiZ5WvLsZ%2F72xylOI4Zg%3D%3D'.strip())

start_month = '2022-10'
end_month = '2022-10'

month_list = pd.period_range(
    start=start_month,
    end=end_month,
    freq='M'
).strftime('%Y%m')

params = {
    'serviceKey': serviceKey,
    'admmCd': '0000000000',  # 현재 이 코드로는 데이터 없음
    'srchFrYm': '',
    'srchToYm': '',
    'lv': '4',
    'regSeCd': '1',
    'type': 'JSON',
    'numOfRows': '100',
    'pageNo': '1'
}

df_list = []

with requests.Session() as session:
    for month in month_list:
        params['srchFrYm'] = month
        params['srchToYm'] = month

        page_no = 1
        month_count = 0

        while True:
            params['pageNo'] = str(page_no)

            response = session.get(
                url,
                params=params,
                timeout=30
            )

            if response.status_code != 200:
                raise RuntimeError(
                    f'{month}, {page_no}페이지 조회 실패\n'
                    f'HTTP {response.status_code}\n'
                    f'{response.text[:1000]}'
                )

            data = response.json()
            result = data.get('Response')

            if not isinstance(result, dict):
                raise RuntimeError(
                    f'{month}, {page_no}페이지 응답 구조 오류\n'
                    f'{response.text[:1000]}'
                )

            head = result['head']
            result_code = str(head['resultCode'])

            # 수정: 데이터 없음은 해당 월의 조회를 종료
            if (
                result_code == '3'
                and head.get('resultMsg') == 'NODATA_ERROR'
            ):
                print(
                    f"{month} | 지역코드 {params['admmCd']} | "
                    f'데이터 없음'
                )
                break

            # 그 밖의 API 오류는 실행 중단
            if result_code != '0':
                raise RuntimeError(
                    f'{month}, {page_no}페이지 API 오류: {head}'
                )

            total_count = int(head['totalCount'])

            if total_count == 0:
                print(f'{month}: 데이터 없음')
                break

            items = (result.get('items') or {}).get('item') or []

            if isinstance(items, dict):
                items = [items]

            if not items:
                raise RuntimeError(
                    f'{month}, {page_no}페이지: '
                    f'전체 건수는 {total_count}인데 데이터가 없습니다.'
                )

            df_list.append(pd.DataFrame(items))
            month_count += len(items)

            print(
                f'{month} | 페이지 {page_no} | '
                f'읽은 건수 {month_count:,}/{total_count:,}'
            )

            if month_count >= total_count:
                break

            page_no += 1

# 모든 월과 페이지 합치기
df = (
    pd.concat(df_list, ignore_index=True)
    if df_list
    else pd.DataFrame()
)

# 데이터가 있을 때 인구수 컬럼을 숫자로 변환
if not df.empty:
    quantity_columns = [
        col for col in df.columns
        if col.endswith('NmprCnt')
    ]

    df[quantity_columns] = df[quantity_columns].apply(
        pd.to_numeric,
        errors='raise'
    )

print(f'\n조회 기간: {start_month} ~ {end_month}')
print(f'최종 데이터 크기: {df.shape}')

if not df.empty:
    print('\n월별 데이터 건수:')
    print(df.groupby('statsYm').size())
else:
    print('조회된 데이터가 없습니다. 지역 조회 조건을 확인해야 합니다.')

df.head()

202210 | 지역코드 0000000000 | 데이터 없음

조회 기간: 2022-10 ~ 2022-10
최종 데이터 크기: (0, 0)
조회된 데이터가 없습니다. 지역 조회 조건을 확인해야 합니다.


""


In [19]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from urllib.parse import unquote

serviceKey = unquote('6ZlAcr4RuErZhSgB4zE9YaHbbfiQes2YrtLoy3YvjYZ6D8tTehdXARkpHVMZAJRDumjiZ5WvLsZ%2F72xylOI4Zg%3D%3D'.strip())

code_url = (
    'https://apis.data.go.kr/1741000/StanReginCd'
    '/getStanReginCdList'
)

code_params = {
    'ServiceKey': serviceKey,  # 문서 기준 대문자 S
    'type': 'xml',
    'pageNo': '1',
    'numOfRows': '1000',
    'flag': 'Y'
    # locatadd_nm은 지역 제한을 하지 않도록 생략
}

code_rows = []
page_no = 1

with requests.Session() as session:
    while True:
        code_params['pageNo'] = str(page_no)

        response = session.get(
            code_url,
            params=code_params,
            timeout=60
        )

        if response.status_code != 200:
            raise RuntimeError(
                f'코드 목록 조회 실패: HTTP {response.status_code}\n'
                f'{response.text[:1000]}'
            )

        root = ET.fromstring(response.content)

        result_code = root.findtext('.//resultCode')
        result_msg = root.findtext('.//resultMsg')

        # 문서의 데이터 없음 코드
        if result_code == 'INFO-200':
            print('조회할 코드가 없습니다.')
            break

        if result_code != 'INFO-0':
            raise RuntimeError(
                f'코드 목록 API 오류: {result_code}, {result_msg}\n'
                f'{response.text[:1000]}'
            )

        total_count = int(root.findtext('.//totalCount', default='0'))

        rows = [
            {
                child.tag: child.text
                for child in row
            }
            for row in root.findall('.//row')
        ]

        if not rows:
            if len(code_rows) < total_count:
                raise RuntimeError('전체 건수보다 적게 읽었는데 빈 페이지입니다.')
            break

        code_rows.extend(rows)

        print(
            f'코드 목록 | 페이지 {page_no} | '
            f'{len(code_rows):,}/{total_count:,}건'
        )

        if len(code_rows) >= total_count:
            break

        page_no += 1

df_code = pd.DataFrame(code_rows)

if not df_code.empty:
    region_code_list = (
        df_code['region_cd']
        .dropna()
        .drop_duplicates()
        .tolist()
    )
else:
    region_code_list = []

print(f'\n법정동 코드 수: {len(region_code_list):,}')

df_code.head()

코드 목록 | 페이지 1 | 1,000/20,560건
코드 목록 | 페이지 2 | 2,000/20,560건
코드 목록 | 페이지 3 | 3,000/20,560건
코드 목록 | 페이지 4 | 4,000/20,560건
코드 목록 | 페이지 5 | 5,000/20,560건
코드 목록 | 페이지 6 | 6,000/20,560건
코드 목록 | 페이지 7 | 7,000/20,560건
코드 목록 | 페이지 8 | 8,000/20,560건
코드 목록 | 페이지 9 | 9,000/20,560건
코드 목록 | 페이지 10 | 10,000/20,560건
코드 목록 | 페이지 11 | 11,000/20,560건
코드 목록 | 페이지 12 | 12,000/20,560건
코드 목록 | 페이지 13 | 13,000/20,560건
코드 목록 | 페이지 14 | 14,000/20,560건
코드 목록 | 페이지 15 | 15,000/20,560건
코드 목록 | 페이지 16 | 16,000/20,560건
코드 목록 | 페이지 17 | 17,000/20,560건
코드 목록 | 페이지 18 | 18,000/20,560건
코드 목록 | 페이지 19 | 19,000/20,560건
코드 목록 | 페이지 20 | 20,000/20,560건
코드 목록 | 페이지 21 | 20,560/20,560건

법정동 코드 수: 20,560


,region_cd,sido_cd,sgg_cd,umd_cd,ri_cd,locatjumin_cd,locatjijuk_cd,locatadd_nm,locat_order,locat_rm,locathigh_cd,locallow_nm,adpt_de
0,2717010900,27,170,109,00,2717010900,2717010900,대구광역시 서구 원대동3가,9,None,2717000000,원대동3가,None
1,2720000000,27,200,000,00,2720000000,2720000000,대구광역시 남구,4,None,2700000000,남구,None
2,2717010700,27,170,107,00,2717010700,2717010700,대구광역시 서구 원대동1가,7,None,2717000000,원대동1가,None
3,2717010800,27,170,108,00,2717010800,2717010800,대구광역시 서구 원대동2가,8,None,2717000000,원대동2가,None
4,3020015200,30,200,152,00,3020015200,3020015200,대전광역시 유성구 둔곡동,52,None,3020000000,둔곡동,None


In [20]:
df_code.head(20).to_clipboard(index=False)

In [21]:
import requests
import pandas as pd

# ─────────────────────────────
# 1. 코드 목록 컬럼을 한글로 변경
# ─────────────────────────────
rename_columns = {
    'region_cd': '지역코드',
    'sido_cd': '시도코드',
    'sgg_cd': '시군구코드',
    'umd_cd': '읍면동코드',
    'ri_cd': '리코드',
    'locatjumin_cd': '지역코드_주민',
    'locatjijuk_cd': '지역코드_지적',
    'locatadd_nm': '지역주소명',
    'locat_order': '서열',
    'locat_rm': '비고',
    'locathigh_cd': '상위지역코드',
    'locallow_nm': '최하위지역명',
    'adpt_de': '생성일'
}

df_code = df_code.rename(columns=rename_columns)

# 코드의 앞자리 0을 유지
code_widths = {
    '지역코드': 10,
    '시도코드': 2,
    '시군구코드': 3,
    '읍면동코드': 3,
    '리코드': 2
}

for col, width in code_widths.items():
    df_code[col] = (
        df_code[col]
        .astype('string')
        .str.strip()
        .str.zfill(width)
    )

# 읍면동 수준만 선택
# 시도·시군구 집계 코드와 하위 리 코드는 제외
mask = (
    df_code['읍면동코드'].ne('000')
    & df_code['리코드'].eq('00')
    & df_code['지역코드'].str.fullmatch(r'\d{10}', na=False)
)

df_region = (
    df_code.loc[mask, ['지역코드', '지역주소명']]
    .drop_duplicates(subset='지역코드')
    .reset_index(drop=True)
)

print(f'전체 코드 목록: {len(df_code):,}건')
print(f'조회 대상 읍면동 코드: {len(df_region):,}건')
print(df_region.head())

# ─────────────────────────────
# 2. 조회 기간과 인구 API 설정
# ─────────────────────────────
start_month = '2022-10'
end_month = '2022-10'

month_list = pd.period_range(
    start=start_month,
    end=end_month,
    freq='M'
).strftime('%Y%m')

url = (
    'https://apis.data.go.kr/1741000/admmSexdAgePpltn'
    '/selectAdmmSexdAgePpltn'
)

params = {
    'serviceKey': serviceKey,
    'admmCd': '',
    'srchFrYm': '',
    'srchToYm': '',
    'lv': '4',
    'regSeCd': '1',
    'type': 'JSON',
    'numOfRows': '100',
    'pageNo': '1'
}

df_list = []
query_log = []

# ─────────────────────────────
# 3. 월 → 지역 → 페이지 반복
# ─────────────────────────────
with requests.Session() as session:
    for month in month_list:
        params['srchFrYm'] = month
        params['srchToYm'] = month

        for region_no, (region_cd, region_name) in enumerate(
            df_region.itertuples(index=False, name=None),
            start=1
        ):
            params['admmCd'] = region_cd

            page_no = 1
            region_count = 0

            while True:
                params['pageNo'] = str(page_no)

                response = session.get(
                    url,
                    params=params,
                    timeout=60
                )

                # 인증·호출 한도 등의 오류는 건너뛰지 않음
                if response.status_code != 200:
                    raise RuntimeError(
                        f'{month} | {region_cd} | {region_name}\n'
                        f'HTTP {response.status_code}\n'
                        f'{response.text[:1000]}'
                    )

                data = response.json()
                result = data.get('Response')

                if not isinstance(result, dict):
                    raise RuntimeError(
                        f'{month} | {region_cd} | 응답 구조 오류\n'
                        f'{response.text[:1000]}'
                    )

                head = result['head']
                result_code = str(head['resultCode'])

                # 데이터가 없으면 해당 지역의 조회 종료
                if (
                    result_code == '3'
                    and head.get('resultMsg') == 'NODATA_ERROR'
                ):
                    if region_count > 0:
                        raise RuntimeError(
                            f'{month} | {region_cd}: '
                            '일부 페이지를 읽은 뒤 데이터 없음이 반환됐습니다.'
                        )
                    break

                if result_code != '0':
                    raise RuntimeError(
                        f'{month} | {region_cd} | API 오류: {head}'
                    )

                total_count = int(head['totalCount'])

                if total_count == 0:
                    break

                items = (result.get('items') or {}).get('item') or []

                if isinstance(items, dict):
                    items = [items]

                if not items:
                    raise RuntimeError(
                        f'{month} | {region_cd} | 페이지 {page_no}: '
                        '전체 건수는 있으나 항목이 없습니다.'
                    )

                df_list.append(pd.DataFrame(items))
                region_count += len(items)

                print(
                    f'{month} | 지역 {region_no}/{len(df_region)} '
                    f'| {region_name} | 페이지 {page_no} '
                    f'| {region_count:,}/{total_count:,}건'
                )

                if region_count >= total_count:
                    break

                page_no += 1

            query_log.append({
                '기준연월': month,
                '요청지역코드': region_cd,
                '요청지역명': region_name,
                '조회건수': region_count,
                '조회상태': '성공' if region_count else '데이터 없음'
            })

            if region_count == 0:
                print(
                    f'{month} | 지역 {region_no}/{len(df_region)} '
                    f'| {region_name} | 데이터 없음 → 건너뛰기'
                )

# ─────────────────────────────
# 4. 결과 합치기와 숫자 변환
# ─────────────────────────────
df = (
    pd.concat(df_list, ignore_index=True)
    if df_list
    else pd.DataFrame()
)

df_query_log = pd.DataFrame(query_log)

if not df.empty:
    quantity_columns = [
        col for col in df.columns
        if col.endswith('NmprCnt')
    ]

    df[quantity_columns] = df[quantity_columns].apply(
        pd.to_numeric,
        errors='raise'
    )

    # 같은 통·반이 반복 수집됐는지 확인
    key_columns = ['statsYm', 'admmCd', 'tong', 'ban']
    duplicate_count = df.duplicated(
        subset=key_columns,
        keep=False
    ).sum()

    print(f'\n같은 연월·행정동·통·반이 반복된 행: {duplicate_count:,}건')

    # ─────────────────────────
    # 5. 인구 데이터 컬럼 한글화
    # ─────────────────────────
    population_rename = {
        'statsYm': '기준연월',
        'admmCd': '행정기관코드',
        'ctpvNm': '시도',
        'sggNm': '시군구',
        'dongNm': '읍면동',
        'tong': '통',
        'ban': '반',
        'totNmprCnt': '총인구수',
        'maleNmprCnt': '남자인구수',
        'femlNmprCnt': '여자인구수'
    }

    for age in range(0, 101, 10):
        age_label = (
            f'{age}~{age + 9}세'
            if age < 100
            else '100세이상'
        )

        population_rename[f'male{age}AgeNmprCnt'] = f'남자_{age_label}'
        population_rename[f'feml{age}AgeNmprCnt'] = f'여자_{age_label}'

    df = df.rename(columns=population_rename)

    ordered_columns = [
        col for col in population_rename.values()
        if col in df.columns
    ]

    df = df[
        ordered_columns
        + [col for col in df.columns if col not in ordered_columns]
    ]

    print('\n조회된 시도:')
    print(df['시도'].unique())

print(f'\n최종 인구 데이터 크기: {df.shape}')

print('\n조회 상태별 지역·월 수:')
print(df_query_log['조회상태'].value_counts())

df.head()

전체 코드 목록: 20,560건
조회 대상 읍면동 코드: 5,067건
         지역코드           지역주소명
0  2717010900  대구광역시 서구 원대동3가
1  2717010700  대구광역시 서구 원대동1가
2  2717010800  대구광역시 서구 원대동2가
3  3020015200   대전광역시 유성구 둔곡동
4  2717010400    대구광역시 서구 상리동
202210 | 지역 1/5067 | 대구광역시 서구 원대동3가 | 데이터 없음 → 건너뛰기
202210 | 지역 2/5067 | 대구광역시 서구 원대동1가 | 데이터 없음 → 건너뛰기
202210 | 지역 3/5067 | 대구광역시 서구 원대동2가 | 데이터 없음 → 건너뛰기
202210 | 지역 4/5067 | 대전광역시 유성구 둔곡동 | 데이터 없음 → 건너뛰기
202210 | 지역 5/5067 | 대구광역시 서구 상리동 | 데이터 없음 → 건너뛰기
202210 | 지역 6/5067 | 대구광역시 서구 중리동 | 데이터 없음 → 건너뛰기
202210 | 지역 7/5067 | 대구광역시 서구 이현동 | 데이터 없음 → 건너뛰기
202210 | 지역 8/5067 | 대구광역시 서구 비산동 | 데이터 없음 → 건너뛰기
202210 | 지역 9/5067 | 대구광역시 서구 평리동 | 데이터 없음 → 건너뛰기
202210 | 지역 10/5067 | 대구광역시 서구 내당동 | 데이터 없음 → 건너뛰기
202210 | 지역 11/5067 | 대구광역시 동구 송정동 | 데이터 없음 → 건너뛰기
202210 | 지역 12/5067 | 대구광역시 동구 덕곡동 | 데이터 없음 → 건너뛰기
202210 | 지역 13/5067 | 대구광역시 동구 지묘동 | 데이터 없음 → 건너뛰기
202210 | 지역 14/5067 | 부산광역시 중구 영주동 | 데이터 없음 → 건너뛰기
202210 | 지역 15/5067 | 부산광역시 중구 대창동1가 | 데이터 없음 → 건너뛰기
202210 | 지역 16/5

KeyboardInterrupt: 

In [22]:
import io
import zipfile
import requests
import pandas as pd

# 공식 행정동 코드: 말소코드 포함
code_url = (
    'https://www.mois.go.kr/cmm/fms/FileDown.do'
    '?atchFileId=FILE_00149355_BR8t5D&fileSn=2'
)

response = requests.get(code_url, timeout=60)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    excel_name = next(
        name for name in archive.namelist()
        if 'KIKCD_H' in name.upper()
        and name.lower().endswith('.xlsx')
    )

    df_admm = pd.read_excel(
        io.BytesIO(archive.read(excel_name)),
        dtype=str
    )

df_admm.columns = df_admm.columns.str.strip()

df_admm['행정동코드'] = (
    df_admm['행정동코드'].str.strip().str.zfill(10)
)

df_admm['_생성일'] = pd.to_datetime(
    df_admm['생성일자'].str.strip(),
    format='%Y%m%d',
    errors='coerce'
)

df_admm['_말소일'] = pd.to_datetime(
    df_admm['말소일자'].str.strip(),
    format='%Y%m%d',
    errors='coerce'
)

# 설정한 조회 기간에 존재했던 읍면동 선택
period_start = pd.Timestamp(start_month)
period_end = pd.Timestamp(end_month) + pd.offsets.MonthEnd(0)

mask = (
    df_admm['읍면동명'].fillna('').str.strip().ne('')
    & (
        df_admm['_생성일'].isna()
        | df_admm['_생성일'].le(period_end)
    )
    & (
        df_admm['_말소일'].isna()
        | df_admm['_말소일'].ge(period_start)
    )
)

df_region = df_admm.loc[mask].copy()

df_region['지역주소명'] = (
    df_region[['시도명', '시군구명', '읍면동명']]
    .fillna('')
    .agg(' '.join, axis=1)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

df_region = (
    df_region.rename(columns={'행정동코드': '지역코드'})
    [['지역코드', '지역주소명']]
    .drop_duplicates(subset='지역코드')
    .reset_index(drop=True)
)

print(f'조회 대상 행정동 코드: {len(df_region):,}건')
print(df_region.head(10))

조회 대상 행정동 코드: 3,582건
         지역코드                  지역주소명
0  1111051500        서울특별시 종로구 청운효자동
1  1111053000          서울특별시 종로구 사직동
2  1111054000          서울특별시 종로구 삼청동
3  1111055000          서울특별시 종로구 부암동
4  1111056000          서울특별시 종로구 평창동
5  1111057000          서울특별시 종로구 무악동
6  1111058000          서울특별시 종로구 교남동
7  1111060000          서울특별시 종로구 가회동
8  1111061500  서울특별시 종로구 종로1.2.3.4가동
9  1111063000      서울특별시 종로구 종로5.6가동
